# X-axis PID and CAN speed tuning

Edit the values in the next cell, then run it. The printed table shows the CAN speed that `main.py` would send for each target offset. Values inside the visual dead zone send zero speed.


In [7]:
from algorithm.pid import PID

# Keep these in sync with main.py while tuning.
KP = 0.10
KI = 0.01
KD = 0.05
X_STEPPER_SPEED_SCALE = 20.0
X_STEPPER_MAX_SPEED = 2000
# Positive visual offset (target left) must produce negative X motor speed.
X_STEPPER_DIRECTION = 1
X_VISION_DEADBAND_PIXELS = 15
FRAME_INTERVAL_SECONDS = 1 / 30

# Positive means the target is left of centre; negative means right.
TARGET_OFFSETS_PIXELS = [100, 50, 25, 16, 15, 10, 0, -10, -15, -16, -25, -50, -100]

def x_can_speed(pid_output):
    speed = round(pid_output * X_STEPPER_SPEED_SCALE * X_STEPPER_DIRECTION)
    return max(-X_STEPPER_MAX_SPEED, min(X_STEPPER_MAX_SPEED, speed))

# Static mapping: each row starts with a new controller. This answers
# 'what speed would this pixel error request?' without a derivative kick.
print('STATIC MAPPING (each offset is independent)')
print('offset px | PID output | CAN speed (steps/s)')
print('----------+------------+---------------------')
for offset in TARGET_OFFSETS_PIXELS:
    if abs(offset) <= X_VISION_DEADBAND_PIXELS:
        output = 0.0
    else:
        controller = PID(KP, KI, KD)
        # PID error is equivalent to the image-space target offset.
        output = controller.update(setpoint=0, measured_value=-offset, dt=FRAME_INTERVAL_SECONDS)
    print(f'{offset:>9} | {output:>10.2f} | {x_can_speed(output):>19}')

# Dynamic test: values below represent consecutive video frames. Here KD
# intentionally reacts to how fast the target moves. A sign reversal means
# the derivative term is stronger than the proportional term.
DYNAMIC_OFFSETS_PIXELS = [100, 50, 25, 16, 15, 10, 0, -10, -15, -16, -25, -50, -100]
controller = PID(KP, KI, KD)
print('\nDYNAMIC RESPONSE (consecutive frames; includes derivative)')
print('offset px | PID output | CAN speed (steps/s)')
print('----------+------------+---------------------')
for offset in DYNAMIC_OFFSETS_PIXELS:
    if abs(offset) <= X_VISION_DEADBAND_PIXELS:
        controller.reset()
        output = 0.0
    else:
        output = controller.update(setpoint=0, measured_value=-offset, dt=FRAME_INTERVAL_SECONDS)
    print(f'{offset:>9} | {output:>10.2f} | {x_can_speed(output):>19}')


STATIC MAPPING (each offset is independent)
offset px | PID output | CAN speed (steps/s)
----------+------------+---------------------
      100 |      10.03 |                 201
       50 |       5.02 |                 100
       25 |       2.51 |                  50
       16 |       1.61 |                  32
       15 |       0.00 |                   0
       10 |       0.00 |                   0
        0 |       0.00 |                   0
      -10 |       0.00 |                   0
      -15 |       0.00 |                   0
      -16 |      -1.61 |                 -32
      -25 |      -2.51 |                 -50
      -50 |      -5.02 |                -100
     -100 |     -10.03 |                -201

DYNAMIC RESPONSE (consecutive frames; includes derivative)
offset px | PID output | CAN speed (steps/s)
----------+------------+---------------------
      100 |      10.03 |                 201
       50 |     -69.95 |               -1399
       25 |     -34.94 |               

In [ ]:
from algorithm.pid import PID

# Basic PID behaviour check. The first update deliberately has no
# derivative term, avoiding a start-up derivative spike.
x_axis = PID(1.0, 0.1, 0.01)

out1 = x_axis.update(setpoint=10, measured_value=8, dt=0.1)
out2 = x_axis.update(setpoint=10, measured_value=9, dt=0.1)
out3 = x_axis.update(setpoint=10, measured_value=10, dt=0.1)

print(f'Output 1: {out1:.3f}')
print(f'Output 2: {out2:.3f}')
print(f'Output 3: {out3:.3f}')

assert round(out1, 3) == 2.020
assert round(out2, 3) == 0.930
assert round(out3, 3) == -0.070
print('PID test passed')
